<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c34-header.png" alt="Nextia Learning · Anomaly Detection: Find What Is Unusual" width="100%">

# Unusual text and logs with embeddings

**[Unusual text and logs with embeddings](https://learning.nextia-ai.com/courses/anomaly-detection/m06/unusual-text-and-logs-with-embeddings/)** · Anomaly Detection: Find What Is Unusual · Module 6, lesson 2 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** turn log lines into templates, find the templates that were never seen before, and compare that cheap method with embeddings: the distance of each new message from the known ones.

| | |
|---|---|
| **Time** | About 40 minutes |
| **Needs** | An internet connection for the setup cells (about 2 MB of data, PyTorch: about 200 MB, and a small embedding model: about 91 MB). No account, no GPU. |
| **You should know** | How to run a notebook cell, from [Meet the Larkfield data](https://learning.nextia-ai.com/courses/anomaly-detection/m01/meet-the-larkfield-data/). The lesson of this notebook explains log lines, templates, embeddings and cosine similarity. |
| **You do not need** | A local project, a GPU, or a paid service: the model runs on your CPU. |
| **Tested** | Python 3.12.13 in Jupyter on macOS (Apple silicon), from a fresh start (*Restart and run all*), with the package versions of the setup cell |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/anomaly-detection/m06/unusual-text-and-logs-with-embeddings/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/notebooks/M06-L02-unusual-text-and-logs-with-embeddings/log-embeddings-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 3.1.1, numpy 2.5.4, scipy 1.18.1, scikit-learn 1.9.1, matplotlib 3.11.2, torch 2.15.0, sentence-transformers 6.2.0. It needs an internet connection. The first time, it takes up to a minute. PyTorch is large (about 200 MB): the first time, this cell can take two or three minutes.

You should see one line per package, with the same versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. A restart is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 numpy==2.5.4 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2 torch==2.15.0 sentence-transformers==6.2.0
from importlib.metadata import version
for p in ['pandas', 'numpy', 'scipy', 'scikit-learn', 'matplotlib', 'torch', 'sentence-transformers']:
    print(p, version(p))

## Setup: get the course data

Run the next cell. It downloads the data that this lesson uses (`ticket_api_logs.csv.gz`) into a folder `anomaly-course/data`, and checks each file's checksum. It needs an internet connection and no account. It takes a few seconds.

You should see `Ready:` and the files with their sizes. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `anomaly-course` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course: no real people, cards or payments. [The dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/data/dataset-card.md) describes every table.

In [ ]:
# Setup: download the course data into anomaly-course/data/ and check each file.
# Standard library only.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C34/"
FILES = {
    "ticket_api_logs.csv.gz": "6ac2245f8473731f0d9e3f600aaa523f2b0ae3a701a81b288ad09010d1afdb0c",
}
CODE_FILES = []
HERE = Path.cwd()
# The notebook works in its own folder anomaly-course/, marked with a file, so it never writes into another project.
if Path(".nextia-notebook").exists():          # the cell ran before: start again from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
WORK = Path("anomaly-course").resolve()
(WORK / "data").mkdir(parents=True, exist_ok=True)
(WORK / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, else download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder anomaly-course and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, WORK / "data" / name, expected)
for name in CODE_FILES:
    fetch(name, LABS + name, WORK / name)
os.chdir(WORK)
print("Ready:", ", ".join(f"{n} ({(Path('data') / n).stat().st_size / 1e6:.1f} MB)" for n in FILES))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd`, NumPy as `np` and matplotlib as `plt`. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it, and the chart style.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.precision", 3)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"figure.figsize": (9, 3.2), "axes.spines.top": False, "axes.spines.right": False})

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple, set)):
            return [norm(x) for x in (sorted(v, key=str) if isinstance(v, set) else v)]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. From log lines to templates

A log line is one sentence that a program writes when something happens: "GET /tickets/T-472971 200 in 39 ms". Most lines are the same sentence with other IDs and numbers. A **template** is that sentence with the parts that change replaced by a mask: `<id>` for an ID and `<num>` for a number.

Run the next cell. It reads the help desk's `ticket-api` logs and shows the first five lines with their templates.

In [ ]:
import re, time
logs = pd.read_csv("data/ticket_api_logs.csv.gz", parse_dates=["ts"])
# the parts of a message that change from line to line: ticket, job, request and agent IDs, and numbers
MASKS = [(re.compile(r"\b[TJ]-\d+\b"), "<id>"), (re.compile(r"\bq-[0-9a-f]+\b"), "<id>"),
         (re.compile(r"\b[AK]-\d+\b"), "<id>"), (re.compile(r"\d+(\.\d+)?"), "<num>")]

def template(message):
    """Replace the parts that change (IDs, numbers) with <id> and <num>."""
    for pattern, mask in MASKS:
        message = pattern.sub(mask, message)
    return message

for message in logs.message.head(5):
    print(f"{message:<58} -> {template(message)}")

Now make the template of every line, and count them.

> **Check.** You should see 137,013 lines, 108,502 different messages and 36 templates. The time is a fraction of a second on a laptop.

In [ ]:
t0 = time.perf_counter()
logs["template"] = logs.message.map(template)
template_seconds = time.perf_counter() - t0
print(f"{len(logs):,} lines, {logs.message.nunique():,} different messages, "
      f"{logs.template.nunique()} templates, in {template_seconds:.2f} s")

## 2. New templates

The service's normal is its first two weeks (1 to 14 September 2025). A template that never appeared in those weeks is **new**. Run the next cell.

> **Check.** You should see 33 templates in the first two weeks and 3 new ones, first seen on 2025-09-16, 2025-10-08 and 2025-10-20.

In [ ]:
known_until = pd.Timestamp("2025-09-15")            # the first two weeks are "known"
known = set(logs.template[logs.ts < known_until])
later = logs[logs.ts >= known_until]
new = (later[~later.template.isin(known)]
       .groupby("template").ts.agg(first_seen="min", lines="size").sort_values("first_seen"))
print(len(known), "templates in the first two weeks;", len(new), "new ones after them:")
print(new.to_string())

Three new kinds of message, out of thousands of lines a day. Templates found every one of them, with a few lines of code and no model. Are they all problems? Read the three templates before you go on.

## 3. Embeddings: how far is a new message from the known ones?

Templates can tell "new" from "known", but not "a little new" from "very new". An **embedding** turns a sentence into a list of numbers (here 384) so that sentences with a similar meaning get similar lists. The **cosine similarity** of two lists is 1 when they point the same way and near 0 when they have nothing in common.

Run the next cell. It downloads a small embedding model (about 91 MB, the first time only) and turns every known and every new template into 384 numbers.

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2",
                            revision="1110a243fdf4706b3f48f1d95db1a4f5529b4d41", device="cpu")
known_list = sorted(known)
vectors_known = model.encode(known_list, normalize_embeddings=True)
vectors_new = model.encode(list(new.index), normalize_embeddings=True)
print(vectors_known.shape, vectors_new.shape)

> **Predict.** Which of the three new templates will be the farthest from every known template: the feature flag, the `KeyError`, or the cache pool?

Run the next cell. For each new template, it finds the most similar known template.

> **Check.** You should see cosines of about 0.34 (feature flag), 0.80 (`KeyError`) and 0.49 (cache pool). The known templates have a median of about 0.79 to their nearest known neighbour.

In [ ]:
similarity = vectors_new @ vectors_known.T     # cosine similarity: every vector has length 1
nearest = pd.DataFrame({"new template": new.index,
                        "nearest known template": [known_list[i] for i in similarity.argmax(axis=1)],
                        "cosine": similarity.max(axis=1).round(3)})
print(nearest.to_string(index=False))
known_to_known = vectors_known @ vectors_known.T
np.fill_diagonal(known_to_known, -1)                # a template is not its own neighbour
known_median = round(float(np.median(known_to_known.max(axis=1))), 3)
print("Known templates: median cosine to their nearest known neighbour:", known_median)

The `KeyError` is close to a known message (another exception of the same function), so by distance it looks almost normal. The feature flag is the farthest from everything, and it is harmless: the team turned on a new reply editor. Distance measures how **different** a message is, not how **bad** it is.

Run the next cell to see the templates on a map. The 384 numbers are squeezed into 2 with PCA, so the picture is only a rough guide.

In [ ]:
from sklearn.decomposition import PCA
xy = PCA(n_components=2, random_state=0).fit_transform(np.vstack([vectors_known, vectors_new]))
k = len(known_list)
plt.figure(figsize=(7, 5))
plt.scatter(xy[:k, 0], xy[:k, 1], color="0.6", label="known (first two weeks)")
plt.scatter(xy[k:, 0], xy[k:, 1], color="tab:blue", s=80, label="new")
for (x, y), name in zip(xy[k:], new.index):
    plt.annotate(name[:32], (x, y), fontsize=8, xytext=(4, 4), textcoords="offset points")
plt.legend(); plt.title("Log templates, 384 numbers each, drawn in 2-D"); plt.show()

## 4. What each method costs

Run the next cell. It times the embedding of 2,000 log lines and compares it with the templates.

In [ ]:
sample = logs.message.sample(2000, random_state=0).tolist()
t0 = time.perf_counter()
model.encode(sample, batch_size=64)
per_line = (time.perf_counter() - t0) / len(sample)
print(f"templates:  {template_seconds / len(logs) * 1e6:.1f} microseconds a line")
print(f"embeddings: {per_line * 1e6:.0f} microseconds a line, "
      f"about {per_line * len(logs) / 60:.1f} minutes for all {len(logs):,} lines")

Both are cheap at this size. At millions of lines a day, the template is thousands of times cheaper, and it needs no model to install or to keep up to date. A good order is: templates first, then embeddings only for the few new templates, to rank them for a person.

## 5. Check yourself

**Your turn 1.** Amira's team changed the cache configuration on 2025-10-20 at 10:00. Use the table `new` to find the template first seen after that time. Store it as `after_change`.

In [ ]:
after_change = ...   # your answer: a template (text)

In [ ]:
expect_hash('the template after the configuration change', after_change, 'be31480967289203')

**Your turn 2.** Use the table `nearest` to find the new template with the **lowest** cosine to its nearest known template: the most unusual by embeddings. Store it as `most_unusual`. Is it a problem?

In [ ]:
most_unusual = ...   # your answer: a template (text)

In [ ]:
expect_hash('the most unusual new template by embeddings', most_unusual, '3ce31481ec6bb883')

In the solution: the configuration change brought `cache pool exhausted, request <id> waited <num> ms for a connection`, and embeddings rank `feature flag new-reply-editor is on for <num>% of agents` as the most unusual, a harmless change. Templates found all three new messages; embeddings added an order, and the order was not "most dangerous first".

## Recap

- A **template** is a log message with its IDs and numbers masked. A template never seen in the normal weeks is a new kind of message. It costs microseconds a line and needs no model.
- An **embedding** turns text into numbers so that similar meanings are close. The cosine to the nearest known message says how different a new message is.
- Different is not dangerous: the harmless feature flag was the farthest from everything, and the real error was close to a known one. Use templates first; use embeddings to group and rank the few new ones for a person.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Where an LLM helps, and where it does not](https://learning.nextia-ai.com/courses/anomaly-detection/m06/where-an-llm-helps/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/anomaly-detection/m06/unusual-text-and-logs-with-embeddings/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.